## 1. CONFIGURACIÓN Y CARGA DE PARÁMETROS

In [0]:
import traceback
import os
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, current_timestamp, lit

try:
    # 1.1 DEFINIR WIDGETS
    # Formato: text(nombre_variable, valor_por_defecto, etiqueta_visual)
    dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
    dbutils.widgets.text("bronze_schema", "sales_bronze", "Bronze Schema Name")
    dbutils.widgets.text("bronze_table", "sales_raw", "Bronze Table Name")
    dbutils.widgets.text("process_path", "/Volumes/iowa_sales/sales_bronze/process", "Source Process Path")
    dbutils.widgets.text("processed_path", "/Volumes/iowa_sales/sales_bronze/processed", "Source Processed Path")

    # 1.2 OBTENER VALORES 
    # Tomará el valor del widget manual en pantalla, o el parámetro inyectado si lo llama un Job
    env = {
        "catalog": dbutils.widgets.get("catalog"),
        "schema": dbutils.widgets.get("bronze_schema"),
        "table": dbutils.widgets.get("bronze_table"),
        "path_in": dbutils.widgets.get("process_path"),
        "path_out": dbutils.widgets.get("processed_path")
    }
    
    target_table = f"{env['catalog']}.{env['schema']}.{env['table']}"
    print(f"Configuración cargada. Tabla destino: {target_table}")

except Exception as e:
    print("Error crítico al obtener los parámetros del notebook.")
    print(traceback.format_exc())
    raise e

## 2. MAPEO DE NOMBRES DE COLUMNAS (Diccionario de Renombrado)

In [0]:
# Mapeamos "Nombre sucio del Parquet" -> "nombre_limpio_para_nuestra_tabla"
column_mapping = {
    "Invoice/Item Number": "invoice_line_no",
    "Date": "date",
    "Store Number": "store",
    "Store Name": "name",
    "Address": "address",
    "City": "city",
    "Zip Code": "zipcode",
    "Store Location": "store_location",
    "County Number": "county_number",
    "County": "county",
    "Category": "category",
    "Category Name": "category_name",
    "Vendor Number": "vendor_no",
    "Vendor Name": "vendor_name",
    "Item Number": "itemno",
    "Item Description": "im_desc",
    "Pack": "pack",
    "Bottle Volume (ml)": "bottle_volume_ml",
    "State Bottle Cost": "state_bottle_cost",
    "State Bottle Retail": "state_bottle_retail",
    "Bottles Sold": "sale_bottles",
    "Sale (Dollars)": "sale_dollars",
    "Volume Sold (Liters)": "sale_liters",
    "Volume Sold (Gallons)": "sale_gallons"
}

## 3. LECTURA DE ARCHIVOS Y RENOMBRADO DE COLUMNAS

In [0]:
from functools import reduce
from pyspark.sql.functions import col, current_timestamp

try:
    print(f"Leyendo y unificando archivos Parquet conflictivos desde: {env['path_in']}")
    
    archivos = [f.path for f in dbutils.fs.ls(env['path_in']) if f.name.endswith('.parquet')]
    
    if not archivos:
        raise ValueError("No se encontraron archivos Parquet en la ruta.")
        
    dataframes = []
    
    for archivo in archivos:
        df_temp = spark.read.format("parquet").load(archivo)
        
        # Convertimos absolutamente todo a String
        for c in df_temp.columns:
            df_temp = df_temp.withColumn(c, col(c).cast("string"))
            
        dataframes.append(df_temp)
        
    # Unificamos esquemas dinámicamente
    df_raw = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dataframes)
    
    # Renombramos
    for old_col, new_col in column_mapping.items():
        if old_col in df_raw.columns:
            df_raw = df_raw.withColumnRenamed(old_col, new_col)
            
    # Agregamos metadata de ingesta
    df_raw = df_raw.withColumn("bronze_saved_date", current_timestamp())

    print("¡Ingesta y unificación completada con éxito!")

except Exception as e:
    print("Error crítico al leer y unificar los archivos Parquet.")
    raise e

## 4. ESCRITURA EN CAPA BRONZE

In [0]:
try:
    print(f"Agregando nuevos datos a la tabla {target_table}...")
    
    df_staged.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(target_table)
        
    print("Ingesta completada exitosamente.")

except Exception as e:
    print("Error durante la escritura en capa Bronze.")
    raise e

## 5. ARCHIVADO DE ARCHIVOS PROCESADOS

In [0]:
try:
    date_suffix = datetime.now().strftime("%Y%m%d_%H%M%S") # Agregamos segundos por si corre varias veces al día
    print(f"Archivando {len(archivos)} archivos...")

    for file_path in archivos:
        base_name = os.path.basename(file_path)
        name, ext = os.path.splitext(base_name)
        new_name = f"{name}_{date_suffix}{ext}"
        dest_path = os.path.join(env["path_out"], new_name)
        
        dbutils.fs.mv(file_path, dest_path)
    
    print(f"Archivos movidos exitosamente a {env['path_out']}")

except Exception as e:
    print("Error al archivar los archivos.")
    raise e

dbutils.notebook.exit("Procesamiento de capa Bronze completado.")

In [0]:
# Consulta para ver la distribución de registros limpios vs corruptos
query_corruptos = f"""
    SELECT 
        is_corrupted, 
        COUNT(*) AS total_records 
    FROM {env['catalog']}.{env['schema']}.{env['table']}
    GROUP BY is_corrupted
"""

# Mostrar el resultado en pantalla
display(spark.sql(query_corruptos))